# 3di-men's own model as the API server (no Hunyuan3D, no Shap-E)

This replaces `hunyuan3d_api_server.ipynb` -- that notebook served Shap-E
(`mode: "direct"`) and real Hunyuan3D-2.1 (`mode: "image"`). This one serves
**only 3di-men's own trained model** (`ml/threedimen/threedimen/model/`), a
small point-cloud autoencoder + text encoder trained on the dataset that
notebook helped collect.

**Important, read before running:**
- **No GPU needed.** Set Accelerator to **None** in the settings panel --
  this model is tiny (~1MB checkpoint) and runs on CPU in well under a
  second per prompt. This costs zero Kaggle GPU quota.
- **Quality expectation stays exactly what it was locally**: this is a
  proof-of-concept trained on tens-to-hundreds of samples. It does not
  generalize the way Hunyuan3D/Shap-E do -- expect outputs that vaguely
  echo the training set's shape distribution, not a recognizable object for
  an arbitrary new prompt. Swapping the serving backend doesn't change what
  the model actually learned.
- **You must upload your trained checkpoint to Kaggle first** -- Kaggle
  can't reach your local disk. Steps:
  1. On your machine: `ml/threedimen/checkpoints/threedimen_v1.pt` (or
     wherever `3dimen model train --out ...` wrote it).
  2. On kaggle.com: **Datasets -> New Dataset -> Upload** that one `.pt`
     file, give it any name (e.g. "threedimen-checkpoint"), create it.
  3. In this notebook: right sidebar -> **Add Input** -> search for and
     attach that dataset. It'll show up under `/kaggle/input/<your-dataset-name>/`.
- Retraining on more data later (e.g. once the 100-sample real-Hunyuan3D
  batch finishes) means re-uploading a new version of that Kaggle Dataset
  and restarting this notebook -- there's no auto-sync.

## 1. Install the small set of packages this needs

In [ ]:
# torch/numpy/scipy are already on Kaggle's base image. trimesh, fastapi,
# and uvicorn are not -- everything else this notebook needs is stdlib.
!pip install -q trimesh fastapi uvicorn
print("installed")

## 2. Model code

Pasted directly (not pip-installed / git-cloned) so this notebook doesn't
depend on the `3di-men` GitHub repo being public or up to date -- it's a
straight copy of `ml/threedimen/threedimen/model/{vocab,networks}.py`.

In [ ]:
from __future__ import annotations
import re
from dataclasses import dataclass, field

import torch
import torch.nn as nn

_TOKEN_RE = re.compile(r"[a-z0-9]+")
PAD = "<pad>"
UNK = "<unk>"

def tokenize(text: str) -> list:
    return _TOKEN_RE.findall(text.lower())

@dataclass
class Vocabulary:
    token_to_id: dict = field(default_factory=dict)

    def __len__(self) -> int:
        return len(self.token_to_id)

    @property
    def pad_id(self) -> int:
        return self.token_to_id[PAD]

    def encode(self, text: str, max_len: int) -> list:
        unk_id = self.token_to_id[UNK]
        ids = [self.token_to_id.get(tok, unk_id) for tok in tokenize(text)][:max_len]
        ids += [self.pad_id] * (max_len - len(ids))
        return ids

    @classmethod
    def from_dict(cls, d: dict) -> "Vocabulary":
        return cls(token_to_id=dict(d["token_to_id"]))


class TextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.project = nn.Sequential(nn.Linear(embed_dim, latent_dim), nn.ReLU(), nn.Linear(latent_dim, latent_dim))

    def forward(self, token_ids):
        mask = (token_ids != 0).float().unsqueeze(-1)
        embedded = self.embedding(token_ids) * mask
        pooled = embedded.sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
        return self.project(pooled)


class PointCloudDecoder(nn.Module):
    def __init__(self, latent_dim, num_points, hidden_dim):
        super().__init__()
        self.num_points = num_points
        self.mlp = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, num_points * 3),
        )

    def forward(self, latent):
        return self.mlp(latent).view(-1, self.num_points, 3)

print("model code loaded")

## 3. Load your uploaded checkpoint

In [ ]:
import glob

candidates = glob.glob("/kaggle/input/**/*.pt", recursive=True)
assert candidates, (
    "No .pt checkpoint found under /kaggle/input -- did you upload it as a "
    "Kaggle Dataset and attach it via 'Add Input' (see the intro cell)?"
)
CHECKPOINT_PATH = candidates[0]
print(f"using checkpoint: {CHECKPOINT_PATH}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
config = checkpoint["config"]
vocab = Vocabulary.from_dict(checkpoint["vocab"])

text_encoder = TextEncoder(len(vocab), config["embed_dim"], config["latent_dim"], pad_idx=vocab.pad_id)
text_encoder.load_state_dict(checkpoint["text_encoder"])
text_encoder.eval()

shape_decoder = PointCloudDecoder(config["latent_dim"], config["num_points"], config["hidden_dim"])
shape_decoder.load_state_dict(checkpoint["shape_decoder"])
shape_decoder.eval()

print(f"3di-men model loaded -- vocab size {len(vocab)}, trained on this checkpoint's dataset")

## 4. The generation function

In [ ]:
import time
import threading
import trimesh

generation_lock = threading.Lock()

def generate_mesh(prompt: str):
    with generation_lock:
        t0 = time.perf_counter()
        with torch.no_grad():
            tokens = torch.tensor([vocab.encode(prompt, config["max_len"])], dtype=torch.long)
            latent = text_encoder(tokens)
            points = shape_decoder(latent)[0].numpy()
        # Same reconstruction as threedimen.ml.decoder.ConvexHullDecoder --
        # this proof-of-concept model outputs raw points, not a mesh, so the
        # convex hull is what turns them into something exportable.
        mesh = trimesh.Trimesh(vertices=points).convex_hull
        elapsed = time.perf_counter() - t0
        return mesh, {"faces": len(mesh.faces), "watertight": bool(mesh.is_watertight), "generation_time_seconds": elapsed}

# Smoke test
_mesh, _meta = generate_mesh("a compact aluminum bracket with mounting holes")
print(_meta)

## 5. API server (FastAPI) -- same job-queue contract as before, one model, no `mode` field

In [ ]:
from pathlib import Path
import uuid

from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
import uvicorn

OUTPUT_DIR = Path("/kaggle/working/api_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

jobs = {}
jobs_lock = threading.Lock()


def _run_job(job_id: str, prompt: str):
    try:
        mesh, meta = generate_mesh(prompt)
        mesh_id = f"{uuid.uuid4().hex}.stl"
        mesh_path = OUTPUT_DIR / mesh_id
        mesh.export(mesh_path)

        result = {
            "prompt": prompt,
            "part_type": "threedimen-poc",
            "part_type_inferred": False,
            "resolved_description": f"3di-men model generation: \"{prompt}\" ({meta['faces']} faces, watertight={meta['watertight']})",
            "parsed_fields": [],
            "defaulted_fields": [],
            "generation_time_seconds": meta["generation_time_seconds"],
            "mesh_url": f"/mesh/{mesh_id}",
        }
        with jobs_lock:
            jobs[job_id] = {"status": "done", "result": result}
    except Exception as exc:
        with jobs_lock:
            jobs[job_id] = {"status": "error", "error": f"{type(exc).__name__}: {exc}"}


@app.post("/generate")
async def api_generate(request: Request):
    body = await request.json()
    prompt = (body.get("prompt") or "").strip()
    if not prompt:
        raise HTTPException(400, "prompt is empty")
    # note: no "mode" field anymore -- there's only one model now. A client
    # (like Hunyuan3DTeacher) that still sends one is fine, it's just ignored.
    job_id = uuid.uuid4().hex
    with jobs_lock:
        jobs[job_id] = {"status": "pending"}
    threading.Thread(target=_run_job, args=(job_id, prompt), daemon=True).start()
    return {"job_id": job_id}


@app.get("/jobs/{job_id}")
def api_job(job_id: str):
    with jobs_lock:
        job = jobs.get(job_id)
    if job is None:
        raise HTTPException(404, "unknown job id")
    return job


@app.get("/mesh/{mesh_id}")
def api_mesh(mesh_id: str):
    path = OUTPUT_DIR / mesh_id
    if not path.is_file() or path.suffix != ".stl":
        raise HTTPException(404, "not found")
    return FileResponse(path, media_type="model/stl", filename=mesh_id)


@app.get("/health")
def health():
    return {"status": "ok", "model": "threedimen-poc"}


def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=_run, daemon=True)
server_thread.start()
time.sleep(2)
print("API server running on http://localhost:8000 (not reachable yet, see next cell)")

## 6. Expose it publicly via a Cloudflare quick tunnel

Same as the previous notebooks -- free, anonymous, no signup. **Leave this
cell running** for as long as you want the frontend to be able to reach it.

In [ ]:
import subprocess, re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

public_url = None
for line in tunnel_proc.stdout:
    print(line, end="")
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if m and public_url is None:
        public_url = m.group(0)
        print()
        print("=" * 70)
        print(f"PUBLIC BACKEND URL: {public_url}")
        print("Paste this into the 'Backend URL' field in your local web UI.")
        print("=" * 70)
        print()

## Notes

- Every request is a fresh forward pass through a ~1MB CPU model --
  expect well under a second per generation, not minutes. The frontend's
  polling loop will just see "done" on its first or second check.
- To update the model with a better checkpoint later: upload a new version
  of the Kaggle Dataset (or a new dataset), reattach it via Add Input,
  restart this notebook. There's no hot-reload.
- To stop serving: interrupt the tunnel cell and/or the kernel.